In [1]:
# --- repo-root guard ---
# Notebooks live in notebooks/, but all data/code paths are relative to the repo root.
# If launched with the working directory set to notebooks/, step up one level so that
# 'src/', 'data/', 'pride_data/' and 'figures/' resolve correctly. Idempotent / no-op at root.
import os, pathlib
_cwd = pathlib.Path.cwd()
if not (_cwd / 'src').exists() and (_cwd.parent / 'src').exists():
    os.chdir(_cwd.parent)
print('working directory:', pathlib.Path.cwd())


working directory: D:\Projects\nanoPhos_env


In [2]:
# --- output directories (created relative to repo root) ---
from pathlib import Path as _P
for _d in ('figures', 'figures/decoupling'):
    _P(_d).mkdir(parents=True, exist_ok=True)

# nanoPhos - SPEC / enrichment decoupling experiment

Isolates the specific contribution of the nanoPhos
phosphoenrichment relative to the SPEC protein preparation, using a factor-decoupling
design (protein prep x phosphoenrichment):

| Branch | Prep | Enrichment |
|---|---|---|
| **classic** | SPEC | nanoPhos-optimized (Fe(III)-NTA; 200 mM NaCl / 10% ACN load; direct Evotip elution) |
| **hybrid** | SPEC | Agilent-standard AssayMAP Fe(III)-NTA |
| **wCleanup** | in-solution + SDB-RPS + SpeedVac | Agilent-standard |
| **woCleanup** | in-solution, crude lysate, no cleanup | Agilent-standard |

Pairwise contrasts each isolate one factor: **classic-hybrid** = enrichment contribution
(SPEC held constant); **hybrid-wCleanup** = SPEC-prep contribution;
**wCleanup-woCleanup** = cleanup effect; **classic-wCleanup** = full nanoPhos vs a standard
conventional workflow.

Depth metrics per Bekker-Jensen 2020: unique **Class I** localized phosphosites
(loc prob >= 0.75) and **phosphopeptide precursors**, per run (n = 4) across 10-1000 ng
(4 branches x 7 inputs x 4 reps = 112 runs).

### Reproducibility
All heavy preprocessing lives in `src/decoupling_preprocess.py`. This notebook reads the
committed result tables in `data/decoupling_*.csv` by default, so every panel reproduces
instantly. To rebuild the tables from the raw Spectronaut reports, set `REGENERATE = True`
below; the raw precursor + Class I reports (~2.7 GB) are read from the relative path
`pride_data/analysis/suppl_figure2/` (the `analysis/suppl_figure2/` folder of the MassIVE deposit).

In [3]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath('src'))
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from core import _hex_to_rgba
import decoupling_preprocess as dp
importlib.reload(dp)

<module 'decoupling_preprocess' from 'D:\\Projects\\nanoPhos_env\\src\\decoupling_preprocess.py'>

In [4]:
# --- configuration -------------------------------------------------------
REGENERATE  = False                                          # True -> rebuild data/ tables from raw reports
RAW_DIR     = 'pride_data/analysis/suppl_figure2' # raw Spectronaut reports (MassIVE); relative
RESULTS_DIR = 'data'                                         # committed result CSVs (data/decoupling_*.csv)

BRANCH_ORDER = dp.BRANCH_ORDER          # ['classic', 'hybrid', 'wCleanup', 'woCleanup']
INPUT_ORDER  = dp.INPUT_ORDER           # [10, 20, 50, 100, 200, 500, 1000]

# classic = brand deep red (hero); the three conventional branches in cooler tones
BRANCH_COLORS = {
    'classic':   '#8A0000',   # nanoPhos (SPEC + optimized enrichment)
    'hybrid':    '#2C6FB0',   # SPEC + standard Agilent enrichment
    'wCleanup':  '#E1812C',   # in-solution + cleanup + standard
    'woCleanup': '#4DAF6A',   # in-solution, crude + standard
}
BRANCH_LABELS = {
    'classic':   'original nanoPhos',
    'hybrid':    'SPEC + default enrichment',
    'wCleanup':  'In-solution digestion + cleanup + default enrichment',
    'woCleanup': 'In-solution digestion + default enrichment',
}
POINT_COLOR = '#393E46'      # Fig 2 convention for jittered points
GRID_COLOR  = '#F3F2F2'

## Data import & preprocessing

`REGENERATE = False` reads the committed result tables; `True` recomputes them from the
raw reports via `dp.regenerate_all` (reads ~2.7 GB once). Either path yields the same eleven
tidy tables consumed by the figure panels below.

In [5]:
if REGENERATE:
    print(f'REGENERATE=True -> rebuilding data/ tables from raw reports in {RAW_DIR} ...')
    tables = dp.regenerate_all(RAW_DIR, RESULTS_DIR)
else:
    tables = dp.load_tables(RESULTS_DIR)

perrun    = tables['decoupling_perrun']              # 112 runs: depth / precursors / selectivity
summary   = tables['decoupling_summary']             # per (branch, input) means +/- SD
contrast  = tables['decoupling_contrasts']           # pairwise isolation contrasts (+ selectivity_diff_pp)
lin_sum   = tables['decoupling_linearity_summary']   # per-branch dilution-linearity summary
lin_sites = tables['decoupling_linearity_sites']     # per-site R^2 / slope (for histograms)
comp_cv   = tables['decoupling_completeness_cv']     # completeness + replicate CV
cv_loo    = tables['decoupling_cv_loo']              # median CV with each replicate left out (Suppl 2e)
overlap   = tables['decoupling_overlap']             # Class I site-identity overlap vs classic
signal    = tables['decoupling_signal']              # phospho signal / dyn-range / loc / GRAVY
signal_pr = tables['decoupling_signal_perrun']       # per-run summed phospho signal (Suppl 2c)
rt_profile= tables['decoupling_rt_profile']          # phospho signal vs retention time

print(f'branches: {sorted(perrun.branch.unique())}')
print(f'inputs:   {sorted(perrun.input_ng.unique())} ng')
print(f'runs:     {len(perrun)}  (expect 112 = 4 branches x 7 inputs x 4 reps)')
summary

branches: ['classic', 'hybrid', 'wCleanup', 'woCleanup']
inputs:   [np.int64(10), np.int64(20), np.int64(50), np.int64(100), np.int64(200), np.int64(500), np.int64(1000)] ng
runs:     112  (expect 112 = 4 branches x 7 inputs x 4 reps)


,branch,input_ng,n,classI_mean,classI_sd,prec_mean,prec_sd,selectivity_mean
0,classic,10,4.0,1666.8,91.3,5094.8,108.4,80.8
1,classic,20,4.0,3677.2,357.2,13148.2,396.2,92.1
2,classic,50,4.0,8581.8,628.5,30396.0,793.4,94.6
3,classic,100,4.0,12263.8,1094.8,42694.0,2117.7,88.7
4,classic,200,4.0,17924.8,230.9,63543.2,350.2,92.7
5,classic,500,4.0,23301.2,195.7,82002.8,393.2,83.6
6,classic,1000,4.0,25549.2,177.0,92239.0,598.6,85.8
7,hybrid,10,4.0,661.8,50.1,1675.8,135.0,17.3
8,hybrid,20,4.0,1838.0,50.1,4948.5,80.0,23.6
9,hybrid,50,4.0,4461.2,618.5,12902.2,1347.1,31.0


## Figures

Panels are built below from the tidy tables above (Fig-2 house style: `plotly_white`,
deep-red brand accent for `classic`, PDF export to `figures/decoupling/`).


In [6]:
# --- shared styling helpers (Fig 2 house style) -------------------------
# classic & hybrid share the SPEC preparation and differ ONLY in the
# phosphoenrichment, so they are the emphasized (solid, bold) pair that
# isolates the nanoPhos enrichment optimization. The in-solution branches
# (wCleanup, woCleanup) are drawn as lighter dashed lines for honest context.
EMPHASIZED = {'classic', 'hybrid'}

def _style(fig, ytitle, xtitle='Protein input (ng)', w=650, h=600, logx=True, yrange=None, legend=True):
    fig.update_layout(template='plotly_white', width=w, height=h, font=dict(size=12),
                      xaxis_title=xtitle, yaxis_title=ytitle,
                      legend_title='Workflow', showlegend=legend,
                      legend=dict(font=dict(size=10), yanchor='top', y=0.99, xanchor='left', x=0.01,
                                  bgcolor='rgba(255,255,255,0.6)'))
    if logx:
        fig.update_xaxes(type='log', tickvals=INPUT_ORDER, ticktext=[str(n) for n in INPUT_ORDER])
    fig.update_xaxes(gridcolor=GRID_COLOR)
    fig.update_yaxes(gridcolor=GRID_COLOR, rangemode='tozero' if yrange is None else 'normal',
                     range=yrange)
    return fig

def add_branch_series(fig, df, ycol, agg=True):
    """One series per branch vs input. classic/hybrid (shared SPEC prep) are
    emphasized as solid bold lines; wCleanup/woCleanup as lighter dashed lines.
    agg=True: per-rep points + mean+/-SD line (df = per-run table).
    agg=False: single mean line (df = per branch x input)."""
    for b in BRANCH_ORDER:
        sub = df[df.branch == b]
        if sub.empty:
            continue
        c = BRANCH_COLORS[b]
        emph = b in EMPHASIZED
        present = [n for n in INPUT_ORDER if n in set(sub['input_ng'])]
        if agg:
            fig.add_trace(go.Scatter(
                x=sub['input_ng'], y=sub[ycol], mode='markers', legendgroup=b, showlegend=False,
                marker=dict(size=7, color=c, opacity=0.5 if emph else 0.3,
                            line=dict(width=0.4, color='black'))))
            g = sub.groupby('input_ng')[ycol].agg(['mean', 'std']).reindex(present)
            err = dict(type='data', array=g['std'].fillna(0), visible=True, color=c, thickness=1, width=5)
        else:
            g = sub.set_index('input_ng').reindex(present).rename(columns={ycol: 'mean'})
            err = None
        fig.add_trace(go.Scatter(
            x=present, y=g['mean'], mode='lines+markers', name=BRANCH_LABELS[b], legendgroup=b,
            line=dict(color=c, width=(2.8 if b == 'classic' else 2.2) if emph else 1.5,
                      dash='solid' if emph else 'dash'),
            marker=dict(size=(10 if b == 'classic' else 8) if emph else 6, color=c,
                        line=dict(width=1, color='black')),
            opacity=1.0 if emph else 0.7,
            error_y=err))
    return fig

### Supplementary Figure 2a - Experimental design
Factor-decoupling of the nanoPhos workflow. An unstimulated HeLa lysate dilution series
(1 µg -> 10 ng, n = 4) is processed through four workflows sharing identical LC-MS/MS
(Evosep Eno + Orbitrap Astral Zoom). The two SPEC-based branches (**classic**, **hybrid**)
differ *only* in the phosphoenrichment - nanoPhos-optimized vs the default Agilent AssayMAP
Fe(III)-NTA protocol - so their difference isolates the phospho-specific optimizations
(modified SPEC elution, pre-elution buffer, loading/elution buffers, direct-to-Evotip
elution). The two lower branches replace SPEC with conventional in-solution preparation,
with (**wCleanup**) and without (**woCleanup**) SDB-RPS cleanup. Panel assembled externally
(schematic, panel *a*).

### Supplementary Figure 2b - Class I phosphosite depth vs input
Per-run unique Class I localized phosphosites (points = 4 replicates; line = mean ± SD).
All four workflows shown. The SPEC-based pair (**classic vs hybrid**, solid) isolates the
enrichment optimization: with the *same* SPEC preparation, the nanoPhos-optimized enrichment
roughly doubles depth across the whole range (~1.9-2.5×). The in-solution branches (dashed)
are shown for context - note `woCleanup` (crude lysate) competes on raw depth, which is why
selectivity (panel 2d) is the decisive axis.

In [7]:
fig = go.Figure()
add_branch_series(fig, perrun, 'classI_sites', agg=True)
_style(fig, 'Class I phosphosites')
fig.show()
fig.write_image('figures/decoupling/suppl_figure2b_depth_classI.pdf', width=600, height=600)

### Supplementary Figure 2c - Summed phosphopeptide-precursor intensity vs input
Summed intensity of all phosphorylated precursors per run (points = 4 replicates; line = mean ± SD),
all four workflows. With SPEC preparation held constant (**classic vs hybrid**, solid), the
nanoPhos-optimized enrichment recovers 1.6-3.4× more phospho signal than the default enrichment.
The uncleaned in-solution workflow (`woCleanup`, dashed) yields the highest *raw* signal up to
500 ng (1.1-1.7× classic; it avoids cleanup losses), and `wCleanup` catches up with classic at
1 µg. Raw signal is not selective coverage: read together with panels 2b (Class I depth) and
2d (selectivity), where classic is the only workflow that is both deep and > 80% selective.

In [8]:
# Suppl 2c - summed phospho-precursor signal per run, all four workflows
fig = go.Figure()
add_branch_series(fig, signal_pr, 'phospho_sum', agg=True)
_style(fig, 'Summed phosphopeptide precursor intensity', legend=False)
fig.update_yaxes(type='log', rangemode='normal')
fig.show()
fig.write_image('figures/decoupling/suppl_figure2c_phospho_signal.pdf', width=600, height=600)

### Supplementary Figure 2d - Phosphopeptide selectivity vs input
Phospho precursors / total precursors per run. **classic** is the only workflow that stays
> 80% across the whole range; the default-enrichment SPEC branch (**hybrid**) sits at
~20-35%, and the in-solution branches fail at opposite ends (`wCleanup` at low input,
`woCleanup` at high input). Read together with panel 2b: only the nanoPhos-optimized enrichment
delivers depth *and* selectivity simultaneously.

In [9]:
fig = go.Figure()
add_branch_series(fig, perrun, 'phospho_selectivity_pct', agg=True)
_style(fig, 'Phospho selectivity (%)', yrange=[0, 100], legend=False)
fig.add_hline(y=80, line=dict(color='grey', dash='dot', width=1))
fig.show()
fig.write_image('figures/decoupling/suppl_figure2d_selectivity.pdf', width=600, height=600)

### Supplementary Figure 2e - Replicate precision (median CV) vs input
Median coefficient of variation of Class I phosphosite intensities (sites quantified in all four
replicates; **lower = more reproducible**). Line = median CV across all four replicates; error bars =
range of the median CV recomputed with each replicate left out in turn. nanoPhos (classic) stays < 25% everywhere,
declines steadily with input (~24% -> ~12%) and moves by <= 3 percentage points when any single
replicate is dropped. The irregular SPEC + default-enrichment values (hybrid) are driven by
individual replicates (e.g. 50 ng: 38% -> 22% without well C6; 10 ng: 39% -> 28% without C8),
as is the 1 µg value of the uncleaned in-solution workflow (45% -> 30% without G1). Note that
`woCleanup` has a lower median CV than classic at 10-200 ng, so the message is consistency and
robustness to single replicates, not the lowest CV at every input.

In [10]:
# Suppl 2e - replicate precision (median CV) vs input, all four workflows
# error bars = range of the leave-one-replicate-out median CVs;
# no per-replicate points, since a CV is computed across replicates
fig = go.Figure()
add_branch_series(fig, comp_cv, 'median_CV_pct', agg=False)

loo_range = cv_loo.groupby(['branch', 'input_ng'])['median_CV_pct_without'].agg(['min', 'max'])
for tr in fig.data:
    b = next((k for k, v in BRANCH_LABELS.items() if v == tr.name), None)
    if b is None:
        continue
    med = comp_cv[comp_cv.branch == b].set_index('input_ng')['median_CV_pct'].reindex(tr.x)
    rng = loo_range.loc[b].reindex(tr.x)
    tr.error_y = dict(type='data', symmetric=False, array=(rng['max'] - med).clip(lower=0),
                      arrayminus=(med - rng['min']).clip(lower=0), visible=True,
                      color=BRANCH_COLORS[b], thickness=1, width=5)

_style(fig, 'Median CV (%)', legend=False)
fig.show()
fig.write_image('figures/decoupling/suppl_figure2e_cv.pdf', width=600, height=600)

### Supplementary - quantification/identification quality
Reported for completeness (not a hero panel): replicate CV and data completeness (from the
Class I reports) and per-precursor localization confidence (from the precursor reports).
These are comparable across workflows or reflect classic's greater depth (more marginal
sites), so they are tabulated rather than plotted.

In [11]:
qual = comp_cv.merge(signal[['branch', 'input_ng', 'loc_median', 'loc_pct_ge90',
                             'phospho_sum_mean', 'gravy_pct_hydrophobic']],
                     on=['branch', 'input_ng'], how='left')
qual = qual[['branch', 'input_ng', 'mean_completeness_pct', 'pct_sites_in_all_reps',
             'median_CV_pct', 'loc_median', 'loc_pct_ge90',
             'phospho_sum_mean', 'gravy_pct_hydrophobic']]
qual = qual.sort_values(['branch', 'input_ng']).reset_index(drop=True)
qual.to_csv('data/decoupling_quality_supp.csv', index=False)
qual

,branch,input_ng,mean_completeness_pct,pct_sites_in_all_reps,median_CV_pct,loc_median,loc_pct_ge90,phospho_sum_mean,gravy_pct_hydrophobic
0,classic,10,62.8,33.1,24.4,0.333,37.5,1.000354e+06,10.0
1,classic,20,60.5,30.8,23.1,0.200,31.9,2.735658e+06,10.0
2,classic,50,63.5,34.7,22.0,0.167,34.0,1.047161e+07,10.8
3,classic,100,63.9,35.1,20.9,0.161,35.7,2.103836e+07,11.5
4,classic,200,67.9,42.2,13.6,0.128,38.1,6.342467e+07,11.5
5,classic,500,69.5,44.7,13.9,0.113,39.4,1.396064e+08,12.7
6,classic,1000,69.6,44.7,12.1,0.114,39.8,2.135908e+08,12.8
7,hybrid,10,65.9,37.2,38.8,0.447,41.6,4.003248e+05,7.0
8,hybrid,20,67.6,39.5,20.7,0.334,40.1,1.689603e+06,6.4
9,hybrid,50,61.4,32.2,37.8,0.249,39.2,5.320962e+06,7.3


## Metadata export

Per-panel source data → `MetaInfo_figures_checklist_v04.xlsx` (2a is the schematic, no data).

In [12]:
# === PRIDE MetaInfo export (run after all panels above) ===
import sys; sys.path.insert(0, r"src")
from metainfo_export import dump_panel
SFIG = 2   # figure number (single source of truth for sheet labels)
def _try(fn, sheet):
    try: fn()
    except Exception as e: print(f"  [SKIP {sheet}] {type(e).__name__}: {e}")

# 2a = experimental-design schematic (no data table)
# 2b - Class I phosphosite depth per run (all branches)
_try(lambda: dump_panel(perrun[["branch","input_ng","well","classI_sites"]], f"Suppl Figure {SFIG}b",
    notes={"workflow labels": "; ".join(f"{b} = {BRANCH_LABELS[b]}" for b in BRANCH_ORDER),
           "workflow order": ", ".join(BRANCH_ORDER), "error bars": "mean ± SD of the 4 replicates"}), f"Suppl Figure {SFIG}b")
# 2c - summed phosphopeptide-precursor intensity per run (all branches)
_try(lambda: dump_panel(signal_pr[["branch","input_ng","run","phospho_sum"]], f"Suppl Figure {SFIG}c",
    notes={"workflow labels": "; ".join(f"{b} = {BRANCH_LABELS[b]}" for b in BRANCH_ORDER),
           "workflow order": ", ".join(BRANCH_ORDER), "error bars": "mean ± SD of the 4 replicates"}), f"Suppl Figure {SFIG}c")
# 2d - phosphopeptide selectivity per run (all branches)
_try(lambda: dump_panel(perrun[["branch","input_ng","well","phospho_selectivity_pct"]], f"Suppl Figure {SFIG}d",
    notes={"workflow labels": "; ".join(f"{b} = {BRANCH_LABELS[b]}" for b in BRANCH_ORDER),
           "workflow order": ", ".join(BRANCH_ORDER), "error bars": "mean ± SD of the 4 replicates",
           "reference line": "y = 80 %"}), f"Suppl Figure {SFIG}d")
# 2e - replicate median CV (all branches) + leave-one-replicate-out median CVs
_try(lambda: dump_panel(comp_cv[["branch","input_ng","median_CV_pct"]].merge(
    cv_loo[["branch","input_ng","left_out_well","median_CV_pct_without"]], on=["branch","input_ng"]),
    f"Suppl Figure {SFIG}e",
    notes={"workflow labels": "; ".join(f"{b} = {BRANCH_LABELS[b]}" for b in BRANCH_ORDER),
           "workflow order": ", ".join(BRANCH_ORDER),
           "error bars": "range (min-max) of median_CV_pct_without, i.e. leaving out one replicate at a time"}), f"Suppl Figure {SFIG}e")
print("Suppl Figure 2 export done.")


  [MetaInfo] wrote 'Suppl Figure 2b'  (112 rows x 4 cols, 3 notes)


  [MetaInfo] wrote 'Suppl Figure 2c'  (112 rows x 4 cols, 3 notes)


  [MetaInfo] wrote 'Suppl Figure 2d'  (112 rows x 4 cols, 4 notes)


  [MetaInfo] wrote 'Suppl Figure 2e'  (112 rows x 5 cols, 3 notes)
Suppl Figure 2 export done.
